# 📓 [Day 31 실전 워크북] Cypher 그래프 집계·인덱스 최적화 및 랭킹 추천 엔진 핸즈온

> **학습 목표**:
> 1. `count`, `sum`, `avg`, `min`, `max`, `percentileCont`(중앙값)를 활용한 수치/분위수 요약 집계를 마스터한다.
> 2. `collect`와 `UNWIND`, 리스트 컴프리헨션을 통한 1:N 컬렉션 직렬화/역직렬화를 실습한다.
> 3. `WITH` 다단계 파이프라인 및 `COUNT { }` 서브쿼리로 집계 결과를 정밀 필터링한다.
> 4. RANGE vs TEXT 인덱스와 UNIQUE 제약조건을 생성하고, `EXPLAIN` 실행계획(`NodeIndexSeek` 등)으로 `dbHits` 절감 효과를 검증한다.
> 5. 공유 이웃 기반 추천 랭킹(원점수, 정규화 비율, 최소 지지도 제약)과 그룹별 상위 N개(`collect()[0..N]`) 슬라이싱을 구현한다.

---

In [ ]:
# [환경 설정] Neo4j 연결 및 헬퍼 함수 정의
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(".env", override=True)
load_dotenv("../.env", override=True)
load_dotenv("내작업폴더/day28_Neo4j_설치_Movies/.env", override=True)

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "test0011")
AURA_URI = os.getenv("AURA_URI")
AURA_USER = os.getenv("AURA_USER")
AURA_PASSWORD = os.getenv("AURA_PASSWORD")

driver = None
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print("✅ 로컬 Neo4j 연결 성공:", NEO4J_URI)
except Exception:
    if AURA_URI and AURA_USER and AURA_PASSWORD:
        driver = GraphDatabase.driver(AURA_URI, auth=(AURA_USER, AURA_PASSWORD))
        driver.verify_connectivity()
        print("✅ Neo4j Aura 클라우드 연결 성공:", AURA_URI)
    else:
        raise ConnectionError("Neo4j에 연결할 수 없습니다. .env를 확인하세요.")

def run_cypher(query: str, **params):
    """Cypher 쿼리 실행 후 dict 리스트로 반환하는 공용 헬퍼"""
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

def explain_plan(query: str, profile=False, quiet=False, **params):
    """EXPLAIN/PROFILE 실행계획을 분석하여 (연산자 리스트, total_hits) 반환"""
    with driver.session() as session:
        if profile:
            res = session.run("PROFILE " + query, **params)
            list(res)
            plan = res.consume().profile
        else:
            plan = session.run("EXPLAIN " + query, **params).consume().plan
    total = 0
    ops = []
    def walk(node, depth=0):
        nonlocal total
        if not node: return
        hits = node.get("dbHits")
        total += hits or 0
        name = node.get("operatorType", "").split("@")[0]
        ops.append(name)
        for c in node.get("children", []):
            walk(c, depth + 1)
    walk(plan)
    return ops, total

In [ ]:
# [초기화 및 시드 적재] 실습 격리 네임스페이스 (SmartProduct, SmartBuyer, SmartArtist, SmartSong 등)
run_cypher("MATCH (n:SmartProduct) DETACH DELETE n")
run_cypher("MATCH (n:SmartBuyer) DETACH DELETE n")
run_cypher("MATCH (n:SmartArtist) DETACH DELETE n")
run_cypher("MATCH (n:SmartSong) DETACH DELETE n")
run_cypher("MATCH (n:SmartListener) DETACH DELETE n")

# 1) 스마트 이커머스 상품 & 구매 시드
run_cypher("""
CREATE (p1:SmartProduct {name: '무선이어폰',   code: 'SP01', price: 80000, category: '가전'}),
       (p2:SmartProduct {name: '블루투스스피커', code: 'SP02', price: 60000, category: '가전'}),
       (p3:SmartProduct {name: '노트북거치대',   code: 'SP03', price: 40000, category: '가전'}),
       (p4:SmartProduct {name: '텀블러',       code: 'SP04', price: 20000, category: '리빙'}),
       (p5:SmartProduct {name: '담요',         code: 'SP05', price: 30000, category: '리빙'}),
       (p6:SmartProduct {name: '머그컵',       code: 'SP06', price: 15000, category: '리빙'})

CREATE (b1:SmartBuyer {name: '도윤'}),
       (b2:SmartBuyer {name: '하윤'}),
       (b3:SmartBuyer {name: '지호'}),
       (b4:SmartBuyer {name: '서아'}),
       (b5:SmartBuyer {name: '민재'})

CREATE (b1)-[:PURCHASED {qty: 1}]->(p1),
       (b1)-[:PURCHASED {qty: 2}]->(p4),
       (b2)-[:PURCHASED {qty: 2}]->(p1),
       (b2)-[:PURCHASED {qty: 1}]->(p4),
       (b2)-[:PURCHASED {qty: 1}]->(p2),
       (b3)-[:PURCHASED {qty: 1}]->(p1),
       (b3)-[:PURCHASED {qty: 2}]->(p5),
       (b4)-[:PURCHASED {qty: 3}]->(p6),
       (b4)-[:PURCHASED {qty: 1}]->(p5),
       (b4)-[:PURCHASED {qty: 1}]->(p3),
       (b5)-[:PURCHASED {qty: 1}]->(p1),
       (b5)-[:PURCHASED {qty: 1}]->(p4)
""")

# 2) 스마트 음악 스트리밍 차트 시드
run_cypher("""
CREATE (a1:SmartArtist {name: '루나'}),
       (a2:SmartArtist {name: '제이드'}),
       (a3:SmartArtist {name: '카이'})

CREATE (s1:SmartSong {title: '은하수', tag: ' 발라드|2021 '}),
       (s2:SmartSong {title: '밤하늘', tag: '발라드|2019 '}),
       (s3:SmartSong {title: '파도',   tag: ' 댄스|2022'}),
       (s4:SmartSong {title: '모래성', tag: ' 발라드|2020 '}),
       (s5:SmartSong {title: '등대',   tag: '댄스|2023 '}),
       (s6:SmartSong {title: '질주',   tag: ' 록|2022 '})

CREATE (a1)-[:PERFORMS]->(s1),
       (a1)-[:PERFORMS]->(s2),
       (a2)-[:PERFORMS]->(s3),
       (a2)-[:PERFORMS]->(s4),
       (a2)-[:PERFORMS]->(s5),
       (a3)-[:PERFORMS]->(s6)

CREATE (l1:SmartListener {name: '하늘'}),
       (l2:SmartListener {name: '바다'}),
       (l3:SmartListener {name: '별'}),
       (l4:SmartListener {name: '산'})

CREATE (l1)-[:PLAYED {cnt: 50}]->(s1),
       (l1)-[:PLAYED {cnt: 30}]->(s3),
       (l2)-[:PLAYED {cnt: 40}]->(s1),
       (l2)-[:PLAYED {cnt: 60}]->(s5),
       (l3)-[:PLAYED {cnt: 20}]->(s2),
       (l3)-[:PLAYED {cnt: 45}]->(s3),
       (l3)-[:PLAYED {cnt: 35}]->(s6),
       (l4)-[:PLAYED {cnt: 25}]->(s4),
       (l4)-[:PLAYED {cnt: 55}]->(s5),
       (l4)-[:PLAYED {cnt: 15}]->(s1)
""")
print("🚀 [초기화 및 시드 적재 완료!]")

## 🎯 미션 1. 카테고리별 상품 가격 집계 및 중앙값 (`percentileCont`)
- `SmartProduct`를 카테고리(`category`)별로 묶어 상품 수(`product_cnt`), 총 가격(`total_price`), 평균 가격(`avg_price`), 중앙값(`median_price`)을 구하고, 총 가격 내림차순으로 정렬하세요.

In [ ]:
# [TODO] 미션 1 쿼리 작성
q1 = """
MATCH (p:SmartProduct)
RETURN p.category AS category,
       count(p) AS product_cnt,
       sum(p.price) AS total_price,
       avg(p.price) AS avg_price,
       percentileCont(p.price, 0.5) AS median_price
ORDER BY total_price DESC
"""
res1 = run_cypher(q1)
print("미션 1 결과:", res1)

In [ ]:
# [자가채점] 미션 1 검증
assert len(res1) == 2
assert res1[0]['category'] == '가전' and res1[0]['total_price'] == 180000 and res1[0]['median_price'] == 60000
assert res1[1]['category'] == '리빙' and res1[1]['total_price'] == 65000 and res1[1]['median_price'] == 20000
print("✅ [미션 1 통과!] 그룹핑 집계와 percentileCont 계산이 완벽합니다.")

## 🎯 미션 2. 아티스트별 곡 목록 `collect` 및 문자열 함수 태그 정제
- `SmartArtist`와 그들의 곡(`SmartSong`)을 매칭하여, 아티스트 이름(`artist`), 총 곡 수(`song_cnt`), 그리고 곡 제목 리스트(`titles`), `tag`에서 앞뒤 공백을 제거하고 파이프(`|`) 앞부분만 추출한 장르 리스트(`genres`)를 반환하세요.

In [ ]:
# [TODO] 미션 2 쿼리 작성
q2 = """
MATCH (a:SmartArtist)-[:PERFORMS]->(s:SmartSong)
WITH a, collect(s) AS song_list
RETURN a.name AS artist,
       size(song_list) AS song_cnt,
       [s IN song_list | s.title] AS titles,
       [s IN song_list | trim(split(s.tag, '|')[0])] AS genres
ORDER BY song_cnt DESC, artist ASC
"""
res2 = run_cypher(q2)
print("미션 2 결과:", res2)

In [ ]:
# [자가채점] 미션 2 검증
assert len(res2) == 3
assert res2[0]['artist'] == '제이드' and res2[0]['song_cnt'] == 3
assert '댄스' in res2[0]['genres'] and '발라드' in res2[0]['genres']
print("✅ [미션 2 통과!] collect 및 리스트 컴프리헨션 태그 가공 완료!")

## 🎯 미션 3. 총 판매수량(sum) 3개 이상인 인기 상품 필터링 (WITH + WHERE)
- 고객들의 구매 이력(`PURCHASED`)을 집계하여, 총 판매 수량(`total_qty`)이 3개 이상인 상품의 이름(`product_name`), 카테고리(`category`), 총 판매수량(`total_qty`), 총 매출액(`total_revenue = total_qty * price`)을 구하고 매출액 내림차순으로 정렬하세요.

In [ ]:
# [TODO] 미션 3 쿼리 작성
q3 = """
MATCH (b:SmartBuyer)-[p:PURCHASED]->(pr:SmartProduct)
WITH pr, sum(p.qty) AS total_qty
WHERE total_qty >= 3
RETURN pr.name AS product_name,
       pr.category AS category,
       total_qty,
       total_qty * pr.price AS total_revenue
ORDER BY total_revenue DESC, product_name ASC
"""
res3 = run_cypher(q3)
print("미션 3 결과:", res3)

In [ ]:
# [자가채점] 미션 3 검증
assert len(res3) == 4
assert res3[0]['product_name'] == '무선이어폰' and res3[0]['total_revenue'] == 400000
assert res3[1]['product_name'] == '담요' and res3[1]['total_revenue'] == 90000
print("✅ [미션 3 통과!] WITH 집계 후 필터링 파이프라인이 정상 작동합니다.")

## 🎯 미션 4. UNIQUE 제약조건 및 RANGE 인덱스 생성과 EXPLAIN 검증
- `SmartProduct.code`에 `smart_product_code_unique` UNIQUE 제약을 생성하고, `SmartProduct.name`에 `smart_product_name_idx` 인덱스를 생성한 후 실행계획(`explain_plan`)에서 인덱스 탐색 연산자가 포함되는지 확인하세요.

In [ ]:
# [TODO] 미션 4 제약/인덱스 생성 및 실행계획 분석
run_cypher("CREATE CONSTRAINT smart_product_code_unique IF NOT EXISTS FOR (p:SmartProduct) REQUIRE p.code IS UNIQUE")
run_cypher("CREATE INDEX smart_product_name_idx IF NOT EXISTS FOR (p:SmartProduct) ON (p.name)")
run_cypher("CALL db.awaitIndexes()")

plan_ops, _ = explain_plan("MATCH (p:SmartProduct {name: '무선이어폰'}) RETURN p.code AS code")
print("실행계획 연산자 목록:", plan_ops)

In [ ]:
# [자가채점] 미션 4 검증
constraints = [c['name'] for c in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name")]
assert 'smart_product_code_unique' in constraints
assert any('Index' in op for op in plan_ops), f"Expected Index operator in plan, got {plan_ops}"
print("✅ [미션 4 통과!] UNIQUE 제약조건과 인덱스 실행계획 검증 성공!")

## 🎯 미션 5. 함께 구매(Co-purchased) 상품 추천 랭킹
- `무선이어폰`을 구매한 고객들이 **함께 구매한 다른 상품**을 찾고, 함께 구매한 고객 수(`shared_buyers`)를 기준으로 내림차순 랭킹을 매겨 반환하세요.

In [ ]:
# [TODO] 미션 5 쿼리 작성
q5 = """
MATCH (target:SmartProduct {name: '무선이어폰'})<-[:PURCHASED]-(b:SmartBuyer)-[:PURCHASED]->(other:SmartProduct)
WHERE other <> target
RETURN other.name AS recommended_product,
       count(DISTINCT b) AS shared_buyers,
       other.price AS price
ORDER BY shared_buyers DESC, other.price DESC
"""
res5 = run_cypher(q5)
print("미션 5 추천 결과:", res5)

In [ ]:
# [자가채점] 미션 5 검증
assert len(res5) >= 3
assert res5[0]['recommended_product'] == '텀블러' and res5[0]['shared_buyers'] == 3
assert res5[1]['recommended_product'] == '블루투스스피커' and res5[1]['shared_buyers'] == 1
print("✅ [미션 5 통과!] 공유 이웃 기반 추천 랭킹 알고리즘 정상 작동!")

## 🎯 미션 6. 그룹별 상위 N개 슬라이싱 (아티스트별 1위 곡 추출)
- 아티스트(`SmartArtist`)별로 공연한 곡들의 총 재생수(`total_plays = sum(p.cnt)`)를 계산하여, 아티스트마다 **가장 인기 있는 1위 곡(`best_song`)의 제목과 재생수**를 추출하세요.

In [ ]:
# [TODO] 미션 6 쿼리 작성
q6 = """
MATCH (a:SmartArtist)-[:PERFORMS]->(s:SmartSong)<-[p:PLAYED]-(:SmartListener)
WITH a, s, sum(p.cnt) AS total_plays
ORDER BY a.name ASC, total_plays DESC, s.title ASC
WITH a, collect({title: s.title, plays: total_plays})[0] AS best_song
RETURN a.name AS artist, best_song.title AS hit_song, best_song.plays AS plays
ORDER BY plays DESC
"""
res6 = run_cypher(q6)
print("미션 6 결과:", res6)

In [ ]:
# [자가채점] 미션 6 검증
assert len(res6) == 3
assert res6[0]['artist'] == '제이드' and res6[0]['hit_song'] == '등대' and res6[0]['plays'] == 115
assert res6[1]['artist'] == '루나' and res6[1]['hit_song'] == '은하수' and res6[1]['plays'] == 105
assert res6[2]['artist'] == '카이' and res6[2]['hit_song'] == '질주' and res6[2]['plays'] == 35
print("✅ [미션 6 통과!] 그룹별 상위 N개 collect 슬라이싱 완성!")